# Sabaic OCR YOLO — Google Colab
هذا الـNotebook يشغّل الكود المكتوب داخل المستودع بدون Ultralytics/TRDG أو weights جاهزة.

In [ ]:
!git clone https://github.com/7eaur/Sabaic-OCR-YOLO.git
%cd Sabaic-OCR-YOLO
!pip install -e . --no-deps
!python scripts/check_environment.py

## 1) إضافة الخط
ارفع `NotoSansOldSouthArabian-Regular.ttf` الذي أعطاه الدكتور.

In [ ]:
from google.colab import files
from pathlib import Path
Path('assets/fonts').mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith('.ttf'):
        Path('assets/fonts/NotoSansOldSouthArabian-Regular.ttf').write_bytes(data)
        print('Font ready')


## 2) توليد Synthetic Dataset
ابدأ بعدد صغير للفحص، ثم غيّر الأعداد إلى 10000/1500 أو حسب GPU.

In [ ]:
!python scripts/generate_synthetic.py --train 500 --val 100
!python scripts/validate_labels.py --images data/synthetic/images/train --labels data/synthetic/labels/train --preview-dir outputs/synthetic_previews --preview-count 20


## 3) حساب Anchors من البيانات
انسخ anchors الناتجة إلى `config/model.json` قبل التدريب الكامل.

In [ ]:
!python scripts/fit_anchors.py --images data/synthetic/images/train --labels data/synthetic/labels/train


## 4) Synthetic pretraining

In [ ]:
!python scripts/train_synthetic.py --config config/train_synthetic.json


## 5) Real fine-tuning
**لن يبدأ السكربت إذا كان `data/real/images/train` أقل من 200 صورة حقيقية labeled.** ضع أيضًا validation/test حقيقيين مستقلين.

In [ ]:
!python scripts/validate_labels.py --images data/real/images/train --labels data/real/labels/train --preview-dir outputs/real_label_previews
!python scripts/finetune_real.py --config config/train_real.json


## 6) التقييم النهائي على Real Test

In [ ]:
!python scripts/evaluate.py --checkpoint checkpoints/real_finetune/best.pt


## 7) Inference

In [ ]:
# مثال بعد رفع صورة test.jpg
!python scripts/infer.py --checkpoint checkpoints/real_finetune/best.pt --image test.jpg
